In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


C:\Users\Dhruv Patel\AppData\Local\Temp\ipykernel_21892\2616709328.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki_tool=WikipediaQueryRun(api_wrapper=api_wrapper)

In [3]:
wiki_tool.name

'wikipedia'

In [4]:
from  langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)

vectordb=FAISS.from_documents(documents,embeddings)
retriever=vectordb.as_retriever()
retriever

USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020452DA1910>, search_kwargs={})

In [5]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever, "langsmith_search", "Search for information about LangSmith. For any questions about LangSmith, you must use these tool")


In [6]:
retriever_tool.name

'langsmith_search'

In [8]:
# Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv_tool=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv_tool.name

'arxiv'

In [9]:
tools=[wiki_tool,arxiv_tool,retriever_tool]
print(tools)

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'C:\\Users\\Dhruv Patel\\Downloads\\Savitara_work\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)), ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)), StructuredTool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use these tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000204A427E7A0>, coroutine=<function create_retriever_tool.<locals>.af

In [10]:
from dotenv import load_dotenv
load_dotenv()
from langchain_ollama import ChatOllama
import os

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
# Load Ollama LAMA2 model
llm=ChatOllama(model="llama3.1",temperature=0)


In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [12]:
# Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)

In [13]:
# Agent Executer

from langchain_classic.agents import AgentExecutor
agent_executer=AgentExecutor(agent=agent,tools=tools,verbose=True)

In [24]:
agent_executer.invoke({"input":"What is LangChain?"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'LangChain'}`


evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.com (no credit card required).

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentFall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. Register now →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces 

{'input': 'What is LangChain?',
 'output': 'Based on the tool call response, LangChain is a platform that provides full visibility into LLM (Large Language Model) applications, from individual traces to production-wide performance metrics. It offers observability, evaluation, prompt engineering, and deployment capabilities, and works with many frameworks and providers, including OpenAI, Anthropic, and CrewAI. LangChain also provides a tracing feature that allows users to monitor performance, configure automations, and collect feedback. Additionally, it has an Engine that automatically detects recurring issues in traces, diagnoses their root cause, and resolves them.'}